# L6 Agents Modern

这个版本保留原 notebook 的核心目标：让 Agent 学会调用工具。

但实现上只使用当前版本推荐的两类原语：

- `create_agent`
- `@tool`

同时把原来比较旧、也容易在你本地环境缺包的部分（如旧式 `AgentExecutor`、`PythonREPLTool`）替换成更直接、可读性更高的工具写法。


In [ ]:
import ast
import os
from datetime import date
from urllib.parse import quote

import requests
from dotenv import load_dotenv, find_dotenv

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

_ = load_dotenv(find_dotenv())

# 这里的 llm 是所有 agent 共用的大脑。
# 工具本身不负责“决定什么时候调用”，它们只负责执行具体动作；
# 真正决定“要不要调工具、调哪个工具”的还是这个聊天模型。
llm = ChatOpenAI(
    temperature=0.0,
    model="qwen-max",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)


In [ ]:
@tool
def calculate(expression: str) -> str:
    """Evaluate a simple math expression."""
    try:
        # @tool 的本质是：把普通 Python 函数暴露给 agent 调用。
        # 当模型判断“这个问题更适合用工具算”时，就会构造工具调用参数来执行这个函数。
        #
        # 这里保留最直观的实现方式：直接计算表达式。
        # 生产环境里建议换成更安全的表达式解析器，不要直接 eval 用户输入。
        result = eval(expression)
        return str(result)
    except Exception as exc:
        return f"Calculation error: {exc}"

@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia and return a short summary."""
    # 这个工具展示的是“agent 不只会算数，也能访问外部信息源”。
    # 整个流程可以拆成两步：
    # 1. 先用 opensearch 找最匹配的标题
    # 2. 再用 summary API 拉取该页面摘要
    search_resp = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={
            "action": "opensearch",
            "search": query,
            "limit": 1,
            "namespace": 0,
            "format": "json",
        },
        timeout=20,
    )
    search_resp.raise_for_status()
    payload = search_resp.json()
    titles = payload[1]

    if not titles:
        return f"No Wikipedia page found for: {query}"

    title = titles[0]
    summary_resp = requests.get(
        f"https://en.wikipedia.org/api/rest_v1/page/summary/{quote(title)}",
        timeout=20,
    )
    summary_resp.raise_for_status()
    summary_payload = summary_resp.json()

    extract = summary_payload.get("extract", "No summary available.")
    page_url = summary_payload.get("content_urls", {}).get("desktop", {}).get("page", "")
    return f"Title: {title}\n\nSummary: {extract}\n\nURL: {page_url}"


In [ ]:
# create_agent 返回的是一个可执行的 agent graph。
# 你可以把它理解成“已经把模型 + 工具 + 系统指令组装好的可调用对象”。
general_agent = create_agent(
    model=llm,
    tools=[calculate, search_wikipedia],
    system_prompt=(
        "You are a helpful assistant. "
        "Use tools when they make the answer more accurate."
    ),
    name="general_tool_agent",
)

def print_agent_result(result: dict) -> None:
    # agent.invoke(...) 返回的不是一行最终答案，而是一整个消息状态。
    # 这里把每条消息打印出来，学习时你能观察到完整执行流：
    # user -> assistant 决定调工具 -> tool 返回结果 -> assistant 再组织最终答案
    for message in result["messages"]:
        print(f"[{message.type}] {message.content}")
        print("-" * 80)


In [ ]:
math_result = general_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is 37 * 18 + 5?"}
        ]
    }
)

print_agent_result(math_result)


In [ ]:
wiki_result = general_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Who was Marie Curie?"}
        ]
    }
)

print_agent_result(wiki_result)


In [ ]:
# 原 notebook 里有一个 Python REPL 排序例子。
# 这里把“让 Agent 处理列表排序”保留下来，但换成更安全、也更容易读懂的专用工具。
@tool
def sort_customers(customer_list_literal: str) -> str:
    """Sort a list like [['Last', 'First'], ...] by last name, then first name."""
    try:
        # ast.literal_eval 只解析 Python 字面量，
        # 比直接 eval 更安全，适合把“字符串形式的列表”还原成真正的列表对象。
        rows = ast.literal_eval(customer_list_literal)

        # item 的结构是 [first_name, last_name]，
        # 所以 key=lambda item: (item[1], item[0]) 表示先按姓排序，再按名排序。
        sorted_rows = sorted(rows, key=lambda item: (item[1], item[0]))
        return str(sorted_rows)
    except Exception as exc:
        return f"Sorting error: {exc}"

sorting_agent = create_agent(
    model=llm,
    tools=[sort_customers],
    system_prompt=(
        "You help with structured data tasks. "
        "If sorting is needed, call the tool instead of guessing."
    ),
    name="sorting_agent",
)


In [ ]:
customer_list = [
    ["Harrison", "Chase"],
    ["Lang", "Chain"],
    ["Dolly", "Too"],
    ["Elle", "Elem"],
    ["Geoff", "Fusion"],
    ["Trance", "Former"],
    ["Jen", "Ayai"],
]

sort_question = (
    "Sort these customers by last name and then first name: "
    f"{customer_list}"
)

sort_result = sorting_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": sort_question}
        ]
    }
)

print_agent_result(sort_result)


In [ ]:
@tool
def today_date(_: str = "") -> str:
    """Return today's date in ISO format."""
    # 这个工具很简单，但它很好地说明了：
    # agent 可以把“实时信息获取”外包给工具，而不是靠模型自己猜日期。
    return str(date.today())

time_agent = create_agent(
    model=llm,
    tools=[today_date],
    system_prompt=(
        "You answer date-related questions. "
        "Use the date tool whenever the user asks about today's date."
    ),
    name="time_agent",
)


In [ ]:
date_result = time_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What's the date today?"}
        ]
    }
)

print_agent_result(date_result)
